# Customer Segmentation using RFM and K-Means

This notebook performs simple customer segmentation using Recency, Frequency and Monetary values.


In [ ]:
import pandas as pd

# Load the sales data
sales = pd.read_excel("/content/sales2.xlsx")

print(sales.head())
sales.info()


In [ ]:
# Check missing values in each column
missing_values = sales.isnull().sum()
print(missing_values)


In [ ]:
# Keep rows that have a Customer ID
sales = sales.dropna(subset=["Customer ID"])

# Remove cancelled invoices
sales = sales[~sales["Invoice"].astype(str).str.startswith("C")]

# Keep only positive quantities and prices
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)]

# Remove repeated rows
sales = sales.drop_duplicates()

print("Rows after cleaning:", len(sales))


In [ ]:
# Calculate the amount spent in each transaction
sales["TotalAmount"] = sales["Quantity"] * sales["Price"]

print(sales[["Quantity", "Price", "TotalAmount"]].head())


In [ ]:
# Use the day after the last invoice date as the reference date
reference_date = sales["InvoiceDate"].max() + pd.Timedelta(days=1)

# Create RFM values for each customer
rfm = sales.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda dates: (reference_date - dates.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalAmount", "sum")
)

print(rfm.head())


In [ ]:
from sklearn.preprocessing import StandardScaler

# Select the three RFM columns
rfm_columns = ["Recency", "Frequency", "Monetary"]

# Standardize the values so the three measures are on a similar scale
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[rfm_columns])

print("RFM data has been scaled.")


In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Store the K-Means inertia values
inertia_values = []
cluster_numbers = range(2, 11)

for number in cluster_numbers:
    kmeans = KMeans(n_clusters=number, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertia_values.append(kmeans.inertia_)

# Draw the elbow graph
plt.plot(cluster_numbers, inertia_values, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()


In [ ]:
# Use 2 clusters, as in the original task
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)

# Assign a cluster number to every customer
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

print(rfm.head())


In [ ]:
from sklearn.metrics import silhouette_score

# Calculate the silhouette score
silhouette = silhouette_score(rfm_scaled, rfm["Cluster"])

print("Silhouette Score:", silhouette)


In [ ]:
from sklearn.metrics import davies_bouldin_score

# Calculate the Davies-Bouldin Index
db_score = davies_bouldin_score(rfm_scaled, rfm["Cluster"])

print("Davies-Bouldin Index:", db_score)


In [ ]:
# Visualize the customer groups
plt.figure(figsize=(8, 6))

plt.scatter(
    rfm["Frequency"],
    rfm["Monetary"],
    c=rfm["Cluster"]
)

plt.xlabel("Frequency")
plt.ylabel("Monetary")
plt.title("Customer Clusters")
plt.show()


In [ ]:
# Find the average RFM values for each cluster
cluster_summary = rfm.groupby("Cluster")[rfm_columns].mean()

print(cluster_summary)


## Conclusion

The customers were grouped using their Recency, Frequency and Monetary values. K-Means clustering was used to create customer groups, and the Silhouette Score and Davies-Bouldin Index were calculated to evaluate the clustering result.
